### Assignment - 3

### Imports

In [1]:
import numpy as np 
import pandas as pd 
import nltk
from nltk.corpus import wordnet as wn
from nltk.corpus import semcor as sm
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from gensim.models import KeyedVectors


In [2]:
nltk.download("semcor")
nltk.download("wordnet")
nltk.download("stopwords")

[nltk_data] Downloading package semcor to C:\Users\ramyasri
[nltk_data]     palti\AppData\Roaming\nltk_data...
[nltk_data]   Package semcor is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\ramyasri
[nltk_data]     palti\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\ramyasri
[nltk_data]     palti\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### Data

In [6]:
# load word2vec model and save for future use(saving as it takes lot of time to load from bin file)
model_w2v = KeyedVectors.load_word2vec_format("./GoogleNews-vectors-negative300.bin.gz",binary=True,)
# model_w2v.save("model_w2v.model")
model_w2v.save("model_w2v.wordvectors")

In [3]:
model_w2v = KeyedVectors.load("./model_w2v.wordvectors", mmap='r')

In [4]:
# semcor sentences with both pos and semcor tags
tagged_sents = sm.tagged_sents(tag="both")
sents = sm.sents()

In [9]:
print(tagged_sents[0])

[Tree('DT', ['The']), Tree(Lemma('group.n.01.group'), [Tree('NE', [Tree('NNP', ['Fulton', 'County', 'Grand', 'Jury'])])]), Tree(Lemma('state.v.01.say'), [Tree('VB', ['said'])]), Tree(Lemma('friday.n.01.Friday'), [Tree('NN', ['Friday'])]), Tree('DT', ['an']), Tree(Lemma('probe.n.01.investigation'), [Tree('NN', ['investigation'])]), Tree('IN', ['of']), Tree(Lemma('atlanta.n.01.Atlanta'), [Tree('NN', ['Atlanta'])]), Tree('POS', ["'s"]), Tree(Lemma('late.s.03.recent'), [Tree('JJ', ['recent'])]), Tree(Lemma('primary.n.01.primary_election'), [Tree('NN', ['primary', 'election'])]), Tree(Lemma('produce.v.04.produce'), [Tree('VB', ['produced'])]), Tree(None, ['``']), Tree('DT', ['no']), Tree(Lemma('evidence.n.01.evidence'), [Tree('NN', ['evidence'])]), Tree(None, ["''"]), Tree('IN', ['that']), Tree('DT', ['any']), Tree(Lemma('abnormality.n.04.irregularity'), [Tree('NN', ['irregularities'])]), Tree(Lemma('happen.v.01.take_place'), [Tree('VB', ['took', 'place'])]), Tree(None, ['.'])]


In [5]:
# flatten the tree for each sentence in data
def sub_extract_tags(tag_sent, sent):
    for word in tag_sent:
        if isinstance(word, nltk.tree.Tree):
            sent.append(word.label())
            sub_extract_tags(word, sent)

    return sent

def extract_tags(tag_sent):
    sent = []
    for word in tag_sent:
        temp = []
        if isinstance(word, nltk.tree.Tree):
            temp.append(word.label())
            sub_extract_tags(word, temp)
        temp.append(word.leaves())
        sent.append(temp)
    return sent

cleaned_sent = [extract_tags(tag_sent) for tag_sent in tagged_sents]


In [11]:
# print(len(cleaned_sent))
print(cleaned_sent[0])

[['DT', ['The']], [Lemma('group.n.01.group'), 'NE', 'NNP', ['Fulton', 'County', 'Grand', 'Jury']], [Lemma('state.v.01.say'), 'VB', ['said']], [Lemma('friday.n.01.Friday'), 'NN', ['Friday']], ['DT', ['an']], [Lemma('probe.n.01.investigation'), 'NN', ['investigation']], ['IN', ['of']], [Lemma('atlanta.n.01.Atlanta'), 'NN', ['Atlanta']], ['POS', ["'s"]], [Lemma('late.s.03.recent'), 'JJ', ['recent']], [Lemma('primary.n.01.primary_election'), 'NN', ['primary', 'election']], [Lemma('produce.v.04.produce'), 'VB', ['produced']], [None, ['``']], ['DT', ['no']], [Lemma('evidence.n.01.evidence'), 'NN', ['evidence']], [None, ["''"]], ['IN', ['that']], ['DT', ['any']], [Lemma('abnormality.n.04.irregularity'), 'NN', ['irregularities']], [Lemma('happen.v.01.take_place'), 'VB', ['took', 'place']], [None, ['.']]]


In [12]:
# to see noun tags in corpus
noun_set = set()
for sent in cleaned_sent:
    for word in sent:
        if word[1][0] == 'N' and word[1]!="NE":
            try:
                noun_set.add(word[1])
            except:
                continue
print("Noun tags in corpus:",noun_set)

{'NNS', 'NNPS', 'NNP', 'NN'}


In [6]:
# list of nouns, its semor tags, and pos tag for each sentence
noun_list = []
for sent in cleaned_sent:
    noun_for_sent = []
    for word in sent:
        if word[1] == 'NN' or word[1] == 'NNP' or word[1]=='NNPS' or word[1]=='NNS':
            if len(word[2]) == 1:
                noun_for_sent.append([word[0], word[2][0]])
    noun_list.append(noun_for_sent) 

In [14]:
print(noun_list[:1])

[[[Lemma('friday.n.01.Friday'), 'Friday'], [Lemma('probe.n.01.investigation'), 'investigation'], [Lemma('atlanta.n.01.Atlanta'), 'Atlanta'], [Lemma('evidence.n.01.evidence'), 'evidence'], [Lemma('abnormality.n.04.irregularity'), 'irregularities']]]


### WSD Algorithms

#### Helper functions

In [5]:
# get embedding of a sentence - sent is set of words
def get_embed(sent):
    emb = []
    len_emb = 0
    for word in sent:
        try:
            emb.append(model_w2v.get_vector(word))
            len_emb += 1
        except:
            continue
    # embedding is avergae of word vectors in sent
    emb = np.sum(emb, axis = 0)/len_emb
    return emb

In [6]:
# compute cosine similarity between sense_emb and context_emb
def compute_score(gloss, context_bag):
    
    # get sense embedding from gloss
    sense_emb = get_embed(gloss)

    #get context embedding
    context_embed = get_embed(context_bag)

    # perform cosine score
    return np.dot(sense_emb, context_embed)/(np.linalg.norm(sense_emb)*np.linalg.norm(context_embed))

#### WFS

In [32]:
def wfs(noun, sent):
    try: 
        # create context bag
        word_list = sent.copy()
        word_list.remove(noun)
        context_bag = set(word_list)
        context_bag = context_bag.difference(stopwords.words('english'))

        # get all word senses of sense bag
        syns = wn.synsets(noun)

        # get best sense
        best_sense = syns[0]

        return best_sense
    except:
        return None

#### WFS Performance

In [ ]:
# sents  noun_list cleaned_sent
total_cnt = 0
pred_cnt = 0
for i in range(len(sents)):
    if i%1000 == 0 and i > 0:
        print("Accuracy on tagging ", str(i), "samples is", pred_cnt/total_cnt)

    sent = sents[i]
    nouns = noun_list[i]

    for j in range(len(nouns)):
        try:
            pred_sense = wfs(nouns[j][1], sent)
            actual_sense = nouns[j][0].synset()

            if pred_sense:
                total_cnt += 1
                if pred_sense == actual_sense:
                    pred_cnt += 1
        except:
            continue
print("Accuracy on tagging ", str(i), "samples is", pred_cnt/total_cnt)

#### Extended Lesk

In [7]:
def extended_lesk(noun, sent):
    try: 
        # create context bag
        word_list = sent.copy()
        word_list.remove(noun)
        context_bag = set(word_list)
        context_bag = context_bag.difference(stopwords.words('english'))

        # get all word senses of sense bag
        syns = wn.synsets(noun)

        # get best sense
        best_sense = syns[0]
        max_score = 0

        for syn in syns:
            #get score of gloss of syns
            senses = []
            senses.append(syn)
            senses.extend(syn.hypernyms())
            senses.extend(syn.hyponyms())
            senses.extend(syn.member_holonyms())
            senses.extend(syn.root_hypernyms())

            flag = 0
            for sense in senses:
                gloss = set(word_tokenize(sense.definition())).difference(stopwords.words('english'))
                score = compute_score(gloss, context_bag)

                if score > max_score:
                    max_score = score
                    flag = 1

            if flag == 1:
                best_sense = syn

        return best_sense
    except:
        return None

#### Extended Lesk Performance

In [ ]:
wrong_pred = 0
wrong_flag = 1 # if set, prints wrong predictions too
total_cnt = 0
pred_cnt = 0
for i in range(len(sents)):
    if i%1000 == 0 and i > 0:
        print("Accuracy on tagging ", str(i), "samples is", pred_cnt/total_cnt)

    sent = sents[i]
    nouns = noun_list[i]

    for j in range(len(nouns)):
        try:
            pred_sense = extended_lesk(nouns[j][1], sent)
            actual_sense = nouns[j][0].synset()

            if pred_sense:
                total_cnt += 1
                if pred_sense == actual_sense:
                    pred_cnt += 1
                elif wrong_flag == 1: 
                    if wrong_pred < 10:
                        print("Wrongly Predicted noun:", nouns[j])
                        print(sent)
                        print("actual sense:", actual_sense.definition())
                        print("Predicted sense:", pred_sense.definition())
                    else:
                        break
                    wrong_pred += 1
        except:
            continue
print("Accuracy on tagging ", str(i), "samples is", pred_cnt/total_cnt)

#### MFS

In [ ]:
dict={}
all_nouns=[]
for nouns in noun_list:
    for n in nouns:
        all_nouns.append(n[1])
all_nouns=list(set(all_nouns))
for n in all_nouns:
    dict[n]=[]
for i in range(len(sents)):
    nouns = noun_list[i]
    for j in range(len(nouns)):
        try:
            actual_sense = nouns[j][0].synset()
            dict[nouns[j][1]].append(actual_sense)
        except:
            continue

dict_sense={}
for n in all_nouns:
    try:
        list_senses=dict[n]
        dict_sense[n]=max(set(list_senses),key=list_senses.count)
    except:
        continue

#### MFS Performance

In [ ]:
total_cnt = 0
pred_cnt = 0
for i in range(len(sents)):
    if i%1000 == 0 and i > 0:
        print("Accuracy on tagging ", str(i), "samples is", pred_cnt/total_cnt)

    sent = sents[i]
    nouns = noun_list[i]

    for j in range(len(nouns)):
        try:
            pred_sense = dict_sense[nouns[j][1]]
            actual_sense = nouns[j][0].synset()
            
            if pred_sense == actual_sense:
                pred_cnt+=1
            total_cnt+=1
        except:
            continue

#### Pagerank

In [8]:
def pagerank(word, sent, idx):
	# context 
	context = sent.copy()

	j=0
	end_prev = 0
	start_prev = 0
	word_start=0
	word_end = 0
	l = 2
	start_idx = max(idx-l,0)
	end_idx = min(idx+l+1,len(context))
	dict_nodes={}
	nodes=[]
	li=[]

		
	for i in range(start_idx,end_idx):
		ws = wn.synsets(context[i])
		start_prev_new = j
		for s in ws:
			nodes.append(j)
			dict_nodes[j]={'word':context[i],'sense':s}
			j+=1
		start_prev = start_prev_new
		end_prev = j
		li.append([start_prev,end_prev])
		if context[i] == word :
			word_start=start_prev
			word_end=end_prev
	edges = np.zeros([len(nodes),len(nodes)])
	sum_1 = [0 for i1 in range(len(nodes))]
	rank = [0 for i1 in range(len(nodes))]
	for w in range(len(li)-1):
		for m in range(li[w][0],li[w][1]):
			for n in range(li[w+1][0],li[w+1][1]):
				stopwords_set = set(stopwords.words('english'))
				sense1= set(word_tokenize(dict_nodes[m]['sense'].definition())).difference(stopwords_set)
				sense2= set(word_tokenize(dict_nodes[n]['sense'].definition())).difference(stopwords_set)
				edges[m][n]=compute_score(sense1,sense2)
				sum_1[m]+=edges[m][n]

	d=0.7
	while(True):
		new_rank = [0 for i1 in range(len(nodes))]
		for i1 in range(len(nodes)):
			for i2 in range(len(nodes)):
				if edges[i1][i2]!=0 and sum_1[i2]!=0:
					new_rank[i1]+=((rank[i2]/sum_1[i2])*edges[i1][i2])
			new_rank[i1]=new_rank[i1]*d+(1-d)/len(nodes)
		diff = 0
		for i1 in range(len(rank)):
			diff +=abs(rank[i1]-new_rank[i1])
		if diff < 1e-5:
			break
		rank = new_rank

		
	sense_rank_w=[]
	dict_sense={}
	for i in range(word_start,word_end):
		sense_rank_w.append(rank[i])
		dict_sense[i-word_start]=dict_nodes[i]['sense']
	return dict_sense[np.argmax(np.array(sense_rank_w))]

#### Pagerank Performance

In [ ]:
noun_list = []
for sent in cleaned_sent:
    noun_for_sent = []
    idx = 0
    for word in sent:
        if word[1] == 'NN' or word[1] == 'NNP' or word[1]=='NNPS' or word[1]=='NNS':
            if len(word[2]) == 1:
                noun_for_sent.append([word[0], word[2][0], idx])
        idx += 1
    noun_list.append(noun_for_sent) 

In [ ]:
# sents  noun_list cleaned_sent
total_cnt = 0
pred_cnt = 0
wrong_pred = 0
for i in range(len(sents)):
    if i%1000 == 0 and i > 0:
        print(i)
        print("Accuracy on tagging ", str(i), "samples is", pred_cnt/total_cnt)

    sent = sents[i]
    nouns = noun_list[i]

    for j in range(len(nouns)):
        try:
            pred_sense = pagerank(nouns[j][1], sent, nouns[j][2])
            actual_sense = nouns[j][0].synset()

            if pred_sense:
                total_cnt += 1
                if pred_sense == actual_sense:
                    pred_cnt += 1
                else:
                    if wrong_pred < 10:
                        print("Wrongly Predicted noun:", nouns[j])
                        print(sent)
                        print("actual sense:", actual_sense.definition())
                        print("Predicted sense:", pred_sense.definition())
                    wrong_pred += 1

        except:
            continue

print("Accuracy on tagging ", str(i+1), "samples is", pred_cnt/total_cnt)

#### Precision, Recall

### Inference

In [9]:
print("Choose WSD Algorithm (Extended Lesk, Pagerank), type exit to exit GUI")

while(True):
    algo = input("Algorithm:")
    if algo == "exit":
        print("Exiting...")
        break
    sent = word_tokenize(input("Input sentence: "))
    word = input("Word you want to disambiguate: ")
    if algo == "pagerank":
        index = int(input("Index of chosen word in sentence: "))
        print("Predicted sense:",pagerank(word,sent,index).definition())
    elif algo == "extended lesk":
        print("Predicted sense:",extended_lesk(word,sent).definition())
    else:
        print("Choose among pagerank and extended lesk")

Choose WSD Algorithm (Extended Lesk, Pagerank), type exit to exit GUI
Predicted sense: Synset('spoon.n.01')
Exiting...


In [17]:
array=['bank','financial','institution','friend','works','opened','account','even','gave','lift','car']
sentence = word_tokenize("bank is a financial institution my friend works in a bank he opened me a bank account he even gave me a lift in his a car")
for i in range(0,11):
    print ("Word :", array[i])
    print ("Best sense: ", extended_lesk(array[i], sentence).definition())
    # print ("Best sense: ", "\n",pagerank(array[i], sentence,idx[i]).definition())
    print ('')

Word : bank
Best sense:  put into a bank account

Word : financial
Best sense:  involving financial matters

Word : institution
Best sense:  an organization founded and united for a specific purpose

Word : friend
Best sense:  a person you know well and regard with affection and trust

Word : works
Best sense:  be employed

Word : opened
Best sense:  make the opening move

Word : account
Best sense:  a formal contractual relationship established to provide for regular banking or brokerage or business services

Word : even
Best sense:  the latter part of the day (the period of decreasing daylight from late afternoon until nightfall)

Word : gave
Best sense:  transfer possession of something concrete or abstract to somebody

Word : lift
Best sense:  transportation of people or goods by air (especially when other means of access are unavailable)

Word : car
Best sense:  a motor vehicle with four wheels; usually propelled by an internal combustion engine

